# Poseable ghost

Example of using controllers to grab and pose an object relative to the simulation.

<video controls src="./assets/object_posing.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.jupyter.utilities import make_id_generator
from nanover.openmm import OpenMMSimulation
from nanover.trajectory import FrameData

simulation = OpenMMSimulation.from_xml_path("trypsin_benzamidine.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: poseable ghost")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [3]:
from nanover.mdanalysis import frame_data_to_mdanalysis

universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

structure_atoms = universe.select_atoms("not resname BEN")
molecule_atoms = universe.select_atoms("resname BEN and not name H*")

utilities.selections.update_selection("root", renderer="cartoon")
utilities.selections.update_selection("ligand", renderer="liquorice", particle_ids=universe.select_atoms("resname BEN").atoms.indices)

## Poseable objects

In [4]:
from nanover.utilities.transforms import Transform
from MDAnalysis.lib.transformations import translation_matrix

get_new_object_id = make_id_generator("object.")

OBJECTS = {}

class PoseableObject:
    objects: dict[str, "PoseableObject"] = {}

    @classmethod
    def make_basic(cls, position=(0,0,0)):
        object = cls(
            get_new_object_id(),
            Transform.from_local_to_parent_matrix(translation_matrix(position)),
        )
        cls.objects[object.id] = object
        return object

    @classmethod
    def intersect_all(cls, point):
        for id, object in cls.objects.items():
            if object.contains_point(point):
                return object
        return None

    def __init__(self, id, transform):
        self.id = id
        self.transform = transform
        self.update()
        self.bounds = (np.array((0, 0, 0)), 1)

    def update(self):
        utilities.transforms.update_transform(self.id, transform=self.transform, parent="scene")

    def contains_point(self, point):
        point_local = self.transform.point_parent_to_local(point)
        centroid, radius = self.bounds
        return np.linalg.norm(point_local - centroid) <= radius

In [5]:
import numpy as np
import MDAnalysis as mda


def make_ghost(key, atoms):
    # extract ghost molecule
    ghost_universe = mda.Merge(atoms)
    ghost_positions = ghost_universe.atoms.positions / 10  # angstrom -> nm
    ghost_bond_pairs = ghost_universe.bonds.indices

    object = PoseableObject.make_basic((0, 0, 0))
    centroid = np.mean(ghost_positions, axis=0)
    object.bounds = (centroid, np.linalg.norm(ghost_positions - centroid, axis=0).max())

    # add transparent spheres and lines to scene at positions relative to nanotube in first frame:
    for i, position in enumerate(ghost_positions):
        utilities.objects.update_shape(f"ghost.{key}.{i}", position=position, size=0.1, color=[1.0, 1.0, 1.0, 0.5], parent=object.id)
    for i, (a, b) in enumerate(ghost_bond_pairs):
        utilities.objects.update_line(f"ghost.{key}.{i}", positions=ghost_positions[[a, b]], size=0.05, color=[1.0, 1.0, 1.0, 0.5], parent=object.id)

    return object

Add objects to scene:

In [6]:
ghost_object = make_ghost("molecule", molecule_atoms)

In [7]:
from nanover.jupyter import ImdAgent
from nanover.imd import ParticleInteraction

class Visuals(ImdAgent):
    def update_interactions(self, full_frame: FrameData, frame_update: FrameData):
        ghost_positions = universe.atoms[molecule_atoms.indices].positions / 10  # angstrom -> nm

        target_positions = ghost_object.transform.points_local_to_parent(ghost_positions)
        real_positions = full_frame.particle_positions[molecule_atoms.indices]

        for i, position in enumerate(ghost_positions):
            utilities.objects.update_line(f"targets.{i}", positions=[target_positions[i], real_positions[i]], size=0.01, color=[1.0, 0, 0, 1.0])
            self.interactions.update_interaction(f"targets.{i}", ParticleInteraction(
                position=target_positions[i],
                particles=[int(molecule_atoms.indices[i])],
                type="spring",
                scale=500,
                max_force=100,
            ))

visuals = Visuals.from_runner(imd_runner)
visuals.start()

## Object posing mode

In [8]:
import numpy as np
from nanover.jupyter import Mode

CURSOR_GRABBED_OBJECT: dict[str, PoseableObject] = {}
CURSOR_GRABBED_MATRIX: dict = {}


def compute_cursor_scene_matrix(cursor: dict):
    scale = utilities.scene_transform_scale
    cursor_transform = Transform.from_state_cursor(cursor)
    cursor_in_scene = (
        utilities.scene_transform.parent_to_local_matrix
        @ cursor_transform.local_to_parent_matrix
        @ np.diagflat((-scale, scale, scale, 1))
    )
    return cursor_in_scene


class MoveObjectMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = PoseableObject.intersect_all(next_pos)
        available = hovered not in CURSOR_GRABBED_OBJECT.values() and hovered is not None

        # grab hovered object if not already grabbed
        if button == "primary" and available:
            # cursor matrix relative to scene
            cursor_in_scene = compute_cursor_scene_matrix(cursor)
            # object matrix relative to scene
            object_in_scene = hovered.transform.local_to_parent_matrix
            # matrix transforming cursor to object
            offset_matrix = np.linalg.inv(cursor_in_scene) @ object_in_scene

            CURSOR_GRABBED_OBJECT[key] = hovered
            CURSOR_GRABBED_MATRIX[key] = offset_matrix

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        # release grabbed
        if button == "primary":
            CURSOR_GRABBED_OBJECT.pop(key, None)
            CURSOR_GRABBED_MATRIX.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])

        # if this cursor has grabbed an object, update the object pose from cursor pose
        grabbed = CURSOR_GRABBED_OBJECT.get(key, None)
        if grabbed is not None:
            # cursor matrix relative to scene
            cursor_in_scene = compute_cursor_scene_matrix(cursor)
            # matrix transforming cursor to object
            offset_matrix = CURSOR_GRABBED_MATRIX.get(key, np.identity(4))
            # object matrix relative to scene
            object_in_scene = cursor_in_scene @ offset_matrix

            grabbed.transform = Transform.from_local_to_parent_matrix(object_in_scene)
            grabbed.update()

        # show/remove hover graphic is this cursor hovers something
        hovered = PoseableObject.intersect_all(next_pos)
        if hovered is None:
            utilities.objects.update_shape(f"hovered.{key}")
        else:
            utilities.objects.update_shape(f"hovered.{key}", position=hovered.bounds[0], color=[1.0, 1.0, 0.0, .5], size=.5, parent=hovered.id)


utilities.use_interaction_modes()
utilities.add_interaction_mode(MoveObjectMode, "move object", icon="✊")

In [9]:
utilities.show_logging()

Output()